In [1]:
import pickle
import sys
import CRPS.CRPS as pscore
import numpy as np
from pathlib import Path

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1"


import multiprocessing as mp
mp.set_start_method('spawn')

sys.path.insert(0, '../LSTM_next_activity_duration/notebooks/evaluation/')
sys.path.insert(0, '../../../../Evaluation')

import conduct_evaluation
import normal_evaluation.normal_evaluation
from prefix_duration_predictor import PrefixDurationPredictor, NOTEBOOK_DIR
from normal_evaluation.lstm_evaluation import SampleOutcomes_LSTM


get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
with open('../../../transformed_event_logs/PCR_start_end_test.pickle', 'rb') as f:
    test_data = pickle.load(f)


n_processes = 20
batch_size = 10
N = 1000

In [3]:
event_log_properties = {
    'case_name' : 'case:concept:name',
    'concept_name' : 'concept:name',
    'timestamp_name' : 'time:timestamp_start',
    'time_since_case_start_column' : '',
    'time_since_last_event_column' : '',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 1,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.0,
    'window_size' : 'auto',
    'categorical_columns' : ['concept:name'],
    'continuous_columns' : ['seconds_in_day', 'day_in_week', 'duration_seconds'],
    'continuous_positive_columns' : []
}

#NOTEBOOK_DIR = Path(__file__).resolve().parent
LSTM_ROOT = (NOTEBOOK_DIR / "../..").resolve()
LOADER_DIR = (NOTEBOOK_DIR / "../../../../load/event_log_loader").resolve()
ENCODED_DIR = (NOTEBOOK_DIR / "../../../../load/encoded_data").resolve()
TRANSFORMED_LOG_DIR = (NOTEBOOK_DIR / "../../../../../transformed_event_logs").resolve()
MODEL_DIR = (NOTEBOOK_DIR / "../training_variational_dropout/PCR").resolve()

TRAIN_DATA_PATH = (ENCODED_DIR / "PCR_1_train.pkl").resolve()

selected_cat_attributes = ['concept:name']
selected_num_attributes = ['seconds_in_day', 'day_in_week']

lstm_predictor = PrefixDurationPredictor(
        train_loader_path = TRAIN_DATA_PATH,
        model_dir = MODEL_DIR,
        model_path = None,
        event_log_properties= event_log_properties,
        selected_cat_attributes = selected_cat_attributes,
        selected_num_attributes = selected_num_attributes,
        device = 'cpu'
)

Embeddings:  ModuleList(
  (0): Embedding(9, 16)
)
Total embedding feature size:  16
Input feature size:  18
Cells hidden size:  128
Number of LSTM layer:  2
Dropout rate:  0.1




from pyinstrument import Profiler

prof = Profiler()
prof.start()
try:
    evaluator_A = conduct_evaluation.ConductEvaluation(lstm_predictor, SampleOutcomes_LSTM, {
                                                        },
                                        test_data, n_processes=n_processes, batch_size=batch_size, n=N)
    likelihoods_A = evaluator_A.sample_cases(False, False)
except KeyboardInterrupt:
    print('interrupted - stopping profiling')
finally:
    prof.stop()
    html = prof.output_html()

    # save locally on remote
    with open("profile.html", "w") as f:
        f.write(html)

In [4]:
evaluator_A = conduct_evaluation.ConductEvaluation(lstm_predictor, SampleOutcomes_LSTM, {
    
                                                    },
                                    test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True, False)


  0%|                                                  | 0/1202 [00:00<?, ?it/s]


  0%|                                                  | 0/1202 [00:19<?, ?it/s]


  0%|                                     | 1/1202 [07:57<159:26:41, 477.94s/it]


  1%|▎                                     | 11/1202 [12:51<19:26:03, 58.74s/it]


  3%|█▎                                     | 41/1202 [14:12<4:28:51, 13.89s/it]

...


 82%|███████████████████████████████▊       | 981/1202 [00:04<00:00, 332.60it/s]


 86%|████████████████████████████████▌     | 1031/1202 [00:04<00:00, 226.68it/s]


 91%|██████████████████████████████████▍   | 1091/1202 [00:05<00:00, 248.78it/s]


 98%|█████████████████████████████████████▎| 1181/1202 [00:05<00:00, 334.00it/s]


100%|██████████████████████████████████████| 1202/1202 [00:05<00:00, 227.38it/s]

In [5]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.292595639022766661586627853')

In [6]:
np.mean(get_pscores(likelihoods_A))

np.float64(20946.998536758954)